In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

customer_schema = """
    customer_id STRING,
    email STRING,
    first_name STRING,
    last_name STRING,
    gender STRING,
    street STRING,
    city STRING,
    country_code STRING,
    row_status STRING,
    row_time TIMESTAMP
"""
def batch_upsert(microBatchDF, batchId):
    window = Window.partitionBy("customer_id").orderBy(F.col("row_time").desc())

    (
        microBatchDF.filter(F.col('row_status').isin(["insert", "update"]))
            .withColumn("rank", F.rank().over(window))
            .filter(F.col('rank') == 1)
            .drop("rank")
            .createOrReplaceTempView("ranked_customers")
    )

    sql_query = """
        MERGE INTO dev.silver.customers_silver c
        USING ranked_customers rc
        ON c.customer_id = rc.customer_id
        WHEN MATCHED AND c.row_time < rc.row_time THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """
    microBatchDF.sparkSession.sql(sql_query)

df_country_lookup = spark.read.json("/Volumes/dev/landing_zone/kafka_source/country_lookup/*")
def process_customers_silver():
    (
        spark.readStream
                .table('dev.bookstore_bronze.bookstore_bronze')
                .filter(F.col("topic") == "customers")
                .select(F.from_json(F.col("value").cast("string"), schema=customer_schema).alias('v'))
                .select('v.*')
                .join(
                    F.broadcast(df_country_lookup),
                    F.col("country_code") == F.col("code"),
                    "inner"
                )
            .writeStream
            .foreachBatch(batch_upsert)
            .option("checkpointLocation", "/Volumes/dev/landing_zone/kafka_source/checkpoints/customers_silver")
            .trigger(availableNow=True)
            .start()
    ).awaitTermination()

process_customers_silver()
